# Microdrama Promo Finder (Kuku Assignment - Problem 1)

This notebook demonstrates the end-to-end Promo Finder pipeline:
1. **Parsing** 50 episodes with frame-accurate timestamps.
2. **Candidate Generation** (LLM or deterministic offline baseline).
3. **Validation & Snapping** (enforcing 20-60s window, hook verification, cliffhanger verification).
4. **Heuristic Feature Scoring** (pace, monologue penalty, action density, question cuts).
5. **Global Selection & Deduplication**.
6. **Multi-objective Ranking & Diversity Filtering**.

In [1]:
import sys
from pathlib import Path
import os
import pandas as pd
import json

# Ensure package is in python path
repo_path = Path(".").resolve()
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

from promo_finder.pipeline import run_pipeline
print("Promo Finder package imported successfully!")

Promo Finder package imported successfully!


## 1. Execute the Pipeline

Set `offline=False` and export `GEMINI_API_KEY` to run with Gemini, or run with `offline=True` for a deterministic baseline run.

In [2]:
show_folder = repo_path / "data" / "show"
api_key_set = bool(os.environ.get("GEMINI_API_KEY"))
use_offline = not api_key_set

print(f"Running pipeline (offline={use_offline}, GEMINI_API_KEY set={api_key_set})...")

result = run_pipeline(
    input_folder=show_folder,
    run_id="notebook_run",
    output_dir=repo_path / "outputs",
    config_path=repo_path / "config.yaml",
    offline=use_offline,
    log_callback=print
)

output_dir = Path(result["output_path"])
print(f"\nPipeline completed. Results saved to: {output_dir}")

Running pipeline (offline=True, GEMINI_API_KEY set=False)...
[19:12:06] Promo Finder starting — run_id=notebook_run
[19:12:06] Input folder: /Users/dhruvkumar/Documents/Kuku-Problem1/promo-finder/data/show
[19:12:06] Config: provider=gemini
[19:12:06] Offline baseline enabled; no LLM calls will be made.
[19:12:06] Phase 1: parsed 50 episodes, 3671 events
[19:12:06] Phase 3: Stage 1 returned 250 raw candidates in 0.0s
[19:12:06] Phase 4: valid first pass=250, repaired=0, dropped=0
[19:12:06] Phase 6: removed 0 duplicates, accepted 0 cross-episode combos
[19:12:06] Phase 7: ranked 250 candidates
[19:12:06] Phase 8: outputs written to /Users/dhruvkumar/Documents/Kuku-Problem1/promo-finder/outputs/notebook_run
[19:12:06] Completed in 0.2s

Pipeline completed. Results saved to: /Users/dhruvkumar/Documents/Kuku-Problem1/promo-finder/outputs/notebook_run


## 2. Inspect Validation & Parsing Results

In [3]:
with open(output_dir / "validation_report.json") as f:
    val_data = json.load(f)

print("Validation Summary:")
print(json.dumps(val_data, indent=2))

Validation Summary:
{
  "total_raw": 250,
  "valid_first_pass": 250,
  "repaired": 0,
  "dropped": 0,
  "valid_rate_pct": 100.0,
  "repair_rate_pct": 0.0,
  "drop_rate_pct": 0.0,
  "drop_reasons": {},
  "repair_reasons": {},
  "stage2_duplicates_removed": 0,
  "stage2_combos_raw": 0,
  "stage2_combos_valid": 0,
  "stage2_combo_drop_reasons": {},
  "total_after_stage2": 250
}


## 3. View Top 10 Ranked Promos

In [4]:
df = pd.read_csv(output_dir / "promos.csv")
cols = ["rank", "candidate_id", "duration_s", "hook_quote", "cliffhanger_quote", "final_blended_score"]
df[cols].head(10)

,rank,candidate_id,duration_s,hook_quote,cliffhanger_quote,final_blended_score
0,1,ep17_offline_05,20.7,"Medium close-up tilting up to Juliana's face, ...",As long as I draw breath...,0.4508
1,2,ep15_offline_03,23.3,Low-angle medium shot focusing on one thug rai...,High-angle two-shot as Juliana rushes forward ...,0.4451
2,3,ep02_offline_02,20.7,"I cannot marry you, Juliana.",Tight insert on Juliana's hand clutching the h...,0.4424
3,4,ep35_offline_04,21.8,"Close-up on her trembling, weathered hands gri...",Panning shot from the open doorway revealing t...,0.4411
4,5,ep08_offline_01,21.5,Medium close-up of Juliana seated at a rustic ...,Insert shot of Juliana's hands gently arrangin...,0.4367
5,6,ep19_offline_01,20.5,Full shot of Juliana and Alexander standing si...,"The elder villager steps closer, resting a gen...",0.4355
6,7,ep21_offline_03,21.5,A loud wooden slam as the heavy double barn do...,Wide shot of Lady Juliana and Prudence facing ...,0.4343
7,8,ep03_offline_02,20.7,Look at you...,Close-up of the Lady in Gold Gown beaming triu...,0.4284
8,9,ep14_offline_05,21.8,I found it.,"Close-up on Juliana's pale, tear-streaked face...",0.4272
9,10,ep18_offline_03,20.7,Close-up overhead shot of Juliana asleep on he...,Framed in the doorway engulfed by blazing oran...,0.4260


## 4. Read Top Promo Detail & Posting Rationale

In [5]:
with open(output_dir / "ranked_report.md") as f:
    report_md = f.read()

# Print first 100 lines of markdown report
print("\n".join(report_md.splitlines()[:80]))

# Promo Finder — Ranked Report

Run ID: `notebook_run`
Generated: 2026-11-07 19:12:06
Total candidates ranked: 250

---

## Top 10 Promos

### #1. `ep17_offline_05`
**Segments:** Episode 17 [00:07.00 → 00:27.70]
**Duration:** 20.7s
**Score:** 0.451 (formula: 0.215, rerank: #1)

**🪝 Hook:** The opening screenplay beat creates immediate context.
> "Medium close-up tilting up to Juliana's face, bruised and tear-streaked yet filled with fragile hope."

**🎬 Cliffhanger:** The window ends at the selected final screenplay beat.
> "As long as I draw breath..."

**Why it works:** Deterministic baseline window for pipeline verification.

**Post first because:** Strongest balance of a stop-scrolling hook, standalone conflict, and an unresolved ending in the final shortlist.

Scores — Hook: 3/5, Cliff: 3/5, Standalone: 3/5, Pace: 3/5 | Heuristic: 3.41

Flags: none
Spoiler: med | Context needed: False | Visual risk: high

---

### #2. `ep15_offline_03`
**Segments:** Episode 15 [00:03.50 → 00:26.80]